In [1]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri

import ipywidgets as widgets
from IPython.display import display, clear_output

import seaborn as sns
from tqdm import tqdm
tqdm.pandas()

from importlib import reload

from utils import textutils as tut
tut = reload(tut)

from utils import magutils as mut
mut = reload(mut)

from utils import catutils as catut
catut = reload(catut)

from widgets import save_panel as sp_module
sp_module = reload(sp_module)

from widgets import category_panel as cp_module
cp_module = reload(cp_module)

In [2]:
df = pd.read_pickle("structures_fe_share.pckl")

In [3]:
df.head()

,name,energy,forces,energy_corrected,energy_corrected_per_atom,ase_atoms,mag_mom
0,~/VASP_test/large_volumes/bcc/free_6/0.005,-0.435437,"[[0.0, 0.0, 0.0]]",-0.000022,-0.000022,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.0, -0.001, 0.039]]"
1,~/VASP_test/large_volumes/bcc/free_6/0.100,-0.442888,"[[0.0, 0.0, 0.0]]",-0.007473,-0.007473,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[-0.0, 0.0, 0.226]]"
2,~/VASP_test/large_volumes/bcc/free_6/0.150,-0.577722,"[[0.0, 0.0, 0.0]]",-0.142307,-0.142307,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[-0.0, 0.0, 0.493]]"
3,~/VASP_test/large_volumes/bcc/free_6/0.750,-1.114440,"[[0.0, 0.0, 0.0]]",-0.679025,-0.679025,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.0, -0.0, 1.194]]"
4,~/VASP_test/large_volumes/bcc/free_6/0.850,-0.975302,"[[0.0, 0.0, 0.0]]",-0.539887,-0.539887,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.002, 0.001, 1.065]]"


# 1.Data Preprocessing

## 1.1. Enrich dataset with per-atom and aggregate magnetic/structural properties

In [4]:
df['nat'] = df['ase_atoms'].map(mut.get_n_atoms)
df['volume'] = df['ase_atoms'].map(mut.get_volume)
df['volume_per_atom'] = df['volume'] / df['nat']

df['M'] = df['mag_mom'].map(mut.total_mag_moment)
df['M_per_atom'] = df['M'] / df['nat']
df['M_abs'] = df['mag_mom'].map(mut.total_mag_moment_abs)
df['M_abs_per_atom'] = df['M_abs'] / df['nat']

df['force_max'] = df['forces'].map(mut.max_force)
df['mag_mom_max'] = df['mag_mom'].map(mut.max_mag_mom)

df["all_mag_mom_eq"] = df["mag_mom"].map(mut.all_moments_equal)
df['if_coll'] = df['mag_mom'].map(mut.is_collinear)

### 1.1.1. Check for None/Nan values

In [5]:
nan_locs = df.isna()
if nan_locs.any().any():
    print("⚠️ Found NaNs in the DataFrame:")
    for col in df.columns:
        n_missing = nan_locs[col].sum()
        if n_missing > 0:
            print(f"  - {col}: {n_missing} missing values: {df[col][df[col].isna()].index.tolist()}")
else:
    print("✅ No NaNs found.")

⚠️ Found NaNs in the DataFrame:
  - volume: 15 missing values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
  - volume_per_atom: 15 missing values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]


In [6]:
display(df[df.isna().any(axis=1)])

,name,energy,forces,energy_corrected,energy_corrected_per_atom,ase_atoms,mag_mom,nat,volume,volume_per_atom,M,M_per_atom,M_abs,M_abs_per_atom,force_max,mag_mom_max,all_mag_mom_eq,if_coll
0,~/VASP_test/large_volumes/bcc/free_6/0.005,-0.435437,"[[0.0, 0.0, 0.0]]",-0.000022,-0.000022,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.0, -0.001, 0.039]]",1,NaN,NaN,0.039013,0.039013,0.039013,0.039013,0.0,0.039013,True,True
1,~/VASP_test/large_volumes/bcc/free_6/0.100,-0.442888,"[[0.0, 0.0, 0.0]]",-0.007473,-0.007473,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[-0.0, 0.0, 0.226]]",1,NaN,NaN,0.226000,0.226000,0.226000,0.226000,0.0,0.226000,True,True
2,~/VASP_test/large_volumes/bcc/free_6/0.150,-0.577722,"[[0.0, 0.0, 0.0]]",-0.142307,-0.142307,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[-0.0, 0.0, 0.493]]",1,NaN,NaN,0.493000,0.493000,0.493000,0.493000,0.0,0.493000,True,True
3,~/VASP_test/large_volumes/bcc/free_6/0.750,-1.114440,"[[0.0, 0.0, 0.0]]",-0.679025,-0.679025,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.0, -0.0, 1.194]]",1,NaN,NaN,1.194000,1.194000,1.194000,1.194000,0.0,1.194000,True,True
4,~/VASP_test/large_volumes/bcc/free_6/0.850,-0.975302,"[[0.0, 0.0, 0.0]]",-0.539887,-0.539887,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.002, 0.001, 1.065]]",1,NaN,NaN,1.065002,1.065002,1.065002,1.065002,0.0,1.065002,True,True
5,~/VASP_test/large_volumes/bcc/free_6/1.000,-1.622238,"[[0.0, 0.0, 0.0]]",-1.186823,-1.186823,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.001, -0.0, 1.726]]",1,NaN,NaN,1.726000,1.726000,1.726000,1.726000,0.0,1.726000,True,True
6,~/VASP_test/large_volumes/bcc/free_6/1.250,-1.837182,"[[0.0, 0.0, 0.0]]",-1.401767,-1.401767,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.002, -0.001, 1.934]]",1,NaN,NaN,1.934001,1.934001,1.934001,1.934001,0.0,1.934001,True,True
7,~/VASP_test/large_volumes/bcc/free_6/1.350,-1.989831,"[[0.0, 0.0, 0.0]]",-1.554416,-1.554416,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[-0.0, -0.0, 2.095]]",1,NaN,NaN,2.095000,2.095000,2.095000,2.095000,0.0,2.095000,True,True
8,~/VASP_test/large_volumes/bcc/free_6/1.450,-2.174441,"[[0.0, 0.0, 0.0]]",-1.739026,-1.739026,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.0, 0.0, 2.31]]",1,NaN,NaN,2.310000,2.310000,2.310000,2.310000,0.0,2.310000,True,True
9,~/VASP_test/large_volumes/bcc/free_6/2.250,-2.892608,"[[0.0, 0.0, 0.0]]",-2.457193,-2.457193,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.0, 0.0, 3.292]]",1,NaN,NaN,3.292000,3.292000,3.292000,3.292000,0.0,3.292000,True,True


### 1.1.2. Number of structures with collinear - non collinear margnetic vectors

In [7]:
df['if_coll'].value_counts()

if_coll
True     61299
False     7695
Name: count, dtype: int64

## 1.2. Categorization based on the "name" column values

In [8]:
# For example:
#   - name: "~/VASP_test/large_volumes/bcc/free_6/0.005"
#     → category: "bcc"
#   - name: "/storage/rinalm9z/c_a_AFM/0.8/0.2/2.3773939154..."
#     → category: "c_a"
#
# Categories are assigned as additional boolean columns in the dataframe (multi-label encoding - i.e. one row can belong to multiple categories)
# An extra column "no_cat" marks rows that match none of the defined categories for an extra checkup.

In [9]:
categories_patterns = {
    "bcc": re.compile(r"bcc"),
    "fcc": re.compile(r"fcc"),
    "supercell": re.compile(r"supercell"),
    "phonons": re.compile("phonon|frozen"),
    "bain_path": re.compile("c_a"),
    "hcp": re.compile("hcp"),
    "surface": re.compile("surf"),
    "gamma": re.compile("gamma"),
    "vacancy": re.compile("vacanc"),
    "green_boundary": re.compile(r"S\d+_\d+"),
    "defect": re.compile("defect"),
    "rotation": re.compile("rotat"),
}

# ig
catut.categorize_in_place(df, categories_patterns)

categories = (*categories_patterns.keys(), 'no_cat') # add 'no category' category as an extra checkup (should be zero)

catut.category_count(df, categories)


{'bcc': 29446,
 'fcc': 4198,
 'supercell': 12155,
 'phonons': 2613,
 'bain_path': 13210,
 'hcp': 1706,
 'surface': 1258,
 'gamma': 4340,
 'vacancy': 3198,
 'green_boundary': 975,
 'defect': 11917,
 'rotation': 5226,
 'no_cat': 0}

# 2. Category Panel

In [10]:
features = [
    cp_module.TargetFeature("energy_corrected_per_atom", lambda s: s.to_numpy()),
    cp_module.TargetFeature("forces", lambda s: np.linalg.norm(np.vstack(s), axis=1), is_atom_level=True),
    cp_module.TargetFeature("mag_mom", lambda s: np.linalg.norm(np.vstack(s), axis=1), is_atom_level=True),
    cp_module.TargetFeature("nat", lambda s: s.to_numpy()),
]


In [10]:
panel = cp_module.CategoryPanel(df, categories, features)
panel

CategoryPanel(children=(HBox(children=(VBox(children=(SelectMultiple(description='Categories:', layout=Layout(…

Name: ipywidgets
Version: 8.1.7
Summary: Jupyter interactive widgets
Home-page: http://jupyter.org
Author: Jupyter Development Team
Author-email: jupyter@googlegroups.com
License: BSD 3-Clause License
Location: /opt/homebrew/anaconda3/envs/pyace/lib/python3.10/site-packages
Requires: comm, ipython, jupyterlab_widgets, traitlets, widgetsnbextension
Required-by: 
